<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Reference Agent System Walkthrough</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Reference Agent System Walkthrough

**Purpose:**  
This notebook presents a complete, realistic example of a working AI Agent System.  

Instead of focusing on single concepts, we will walk through an entire agent system from architecture to execution, so you can see how all the pieces (tools, dispatcher, prompt, reasoning, and error handling) come together in practice.

**Real-World Scenario:**  

We will build a **Medical Information Assistant** that can answer general health-related queries responsibly by using multiple tools, showing clear reasoning, and always including medical disclaimers.

**Important Note:**  
This is for educational purposes only. Real medical advice should always come from qualified healthcare professionals.

## 1. System Architecture Overview

Our Reference Medical Agent System follows a clean, modular architecture:

**Core Components:**

- **Tools Layer**: Specialized tools that fetch reliable information (disease/symptom info, drug details, BMI calculation, etc.).
- **Tool Registry**: Central store that holds all available tools for easy management and lookup.
- **Tool Dispatcher**: Safe execution layer that routes requests to the correct tool and handles errors gracefully.
- **LLM + ReAct Prompt**: The reasoning engine that decides the sequence of tool calls.
- **AgentExecutor**: Orchestrates the full multi-step reasoning loop.

**Data Flow:**
1. User asks a health-related question
2. Agent reasons step-by-step and selects appropriate tools
3. Tool Dispatcher executes the tools safely
4. Results are fed back to the agent
5. Agent synthesizes information and provides a final answer with disclaimers

This architecture is widely used in real-world agent systems because it is scalable, maintainable, and auditable.

## 1. Setup

In [1]:
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass
import requests

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph 1.2.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 16.0 which is incompatible.
unclecode-litellm 1.81.13 requires openai>=2.8.0, but you have openai 1.109.1 which is i

Setup complete


In [3]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

## 3. Tool Definitions

We define four meaningful tools for our Medical Information Assistant:

- Real symptom/disease information from Wikipedia
- Basic drug information
- BMI calculation
- Mandatory medical disclaimer tool

In [19]:
@tool
def get_medical_info(condition: str) -> str:
    """
    Fetch reliable medical information about a condition or symptom from Wikipedia.
    """
    try:
        url = f"https://en.wikipedia.org/wiki/{condition.replace(' ', '_')}"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        response = requests.get(url, timeout=10, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        paragraphs = soup.find_all("p")
        summary = ""
        for p in paragraphs:
            text = p.get_text().strip()
            if text and len(text) > 50:
                summary += text + "\n\n"
            if len(summary) >= 800:
                break
        if not summary:
            return f"Could not find reliable information for '{condition}'."
        return f"Information about '{condition}':\n\n" + summary[:800] + "...\n\n(Source: Wikipedia)"
    except Exception as e:
        return f"Could not find reliable information for '{condition}'. Error: {str(e)}"


@tool
def get_drug_info(drug_name: str) -> str:
    """
    Get basic information about a common medication.
    """
    drug_db = {
        "paracetamol": "Used for pain relief and fever reduction. Common brand: Panadol/Tylenol. Max adult dose: 1g per dose, 4g/day.",
        "ibuprofen": "NSAID used for pain, fever, and inflammation. Common brand: Advil, Nurofen. Take with food.",
        "aspirin": "Used for pain, fever, inflammation, and blood thinning. Avoid in children under 16.",
        "amoxicillin": "Antibiotic used for bacterial infections. Requires a prescription.",
        "cetirizine": "Antihistamine used for allergies and hay fever. Common brand: Zyrtec.",
        "omeprazole": "Proton pump inhibitor used for acid reflux and stomach ulcers. Common brand: Prilosec.",
        "metformin": "Used to treat type 2 diabetes. Helps control blood sugar levels.",
        "atorvastatin": "Statin used to lower cholesterol. Common brand: Lipitor.",
    }
    info = drug_db.get(drug_name.lower(), f"Basic information about '{drug_name}' is not available in the demo database.")
    return f"Drug Information for '{drug_name}':\n{info}"


@tool
def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """
    Calculate Body Mass Index (BMI) and provide basic interpretation.
    Provide weight in kg and height in cm as separate numeric values.
    """
    # Guard: if weight_kg was passed as a malformed string, parse it manually
    if isinstance(weight_kg, str):
        import re
        nums = re.findall(r"[\d.]+", str(weight_kg) + "," + str(height_cm))
        if len(nums) >= 2:
            weight_kg = float(nums[0])
            height_cm = float(nums[1])
        else:
            return "Could not parse BMI inputs. Please provide weight_kg and height_cm as numbers."

    bmi = float(weight_kg) / ((float(height_cm) / 100) ** 2)
    if bmi < 18.5:
        category, advice = "Underweight", "Consider consulting a doctor or dietitian."
    elif bmi < 25.0:
        category, advice = "Normal weight", "You are in a healthy weight range."
    elif bmi < 30.0:
        category, advice = "Overweight", "Consider increased physical activity and a balanced diet."
    else:
        category, advice = "Obese", "Consult a healthcare provider for a personalised plan."
    return (
        f"BMI Calculation:\n"
        f"  Weight: {weight_kg} kg | Height: {height_cm} cm\n"
        f"  BMI: {bmi:.1f} ({category})\n"
        f"  Advice: {advice}"
    )

@tool
def medical_disclaimer() -> str:
    """
    Always return a standard medical disclaimer.
    """
    return (
        "MEDICAL DISCLAIMER: This information is for educational purposes only and "
        "does not constitute professional medical advice, diagnosis, or treatment. "
        "Always seek the advice of your physician or other qualified health provider "
        "with any questions you may have regarding a medical condition. "
        "Never disregard professional medical advice or delay in seeking it because "
        "of something you have read here. If you think you may have a medical emergency, "
        "call your doctor or emergency services immediately."
    )


## 4. Tool Registry and Tool Dispatcher

The Tool Registry and Dispatcher form the backbone of our agent system, ensuring safe and organized tool execution.

In [20]:
tool_registry = {
    "get_medical_info": get_medical_info,
    "get_drug_info": get_drug_info,
    "calculate_bmi": calculate_bmi,
    "medical_disclaimer": medical_disclaimer
}

def tool_dispatcher(tool_name: str, **kwargs):
    """
    Safe tool dispatcher with standardized response format.
    """
    if tool_name not in tool_registry:
        return {"status": "error", "tool": tool_name, "error": f"Tool '{tool_name}' not found in registry."}
    try:
        selected_tool = tool_registry[tool_name]
        # LangChain @tool objects expose a .run() method; fall back to direct call
        result = selected_tool.run(kwargs) if hasattr(selected_tool, "run") else selected_tool(**kwargs)
        return {"status": "success", "tool": tool_name, "result": result}
    except Exception as e:
        return {"status": "error", "tool": tool_name, "error": str(e)}


## 5. Full ReAct Agent Setup

## 5a. Guardrails

We add two production-style guardrails **before wiring up the agent**:

### Output Filter
Scans the agent's final answer for unsafe patterns — specific drug dosage instructions,
self-diagnosis claims, and any text that could be mistaken for professional medical advice.
Flagged content is replaced with a safe redaction notice.

### Rate Limiter
Wraps `AgentExecutor.invoke` to enforce:
- A hard cap on **tool calls per request** (default: 10).
- A **per-minute request budget** (default: 5 requests / minute) using a sliding-window counter.

Both guardrails are implemented as plain Python functions/classes so they work with any
LangChain version and are easy to unit-test independently of the LLM.


In [21]:
import re
import time
from collections import deque

# ─────────────────────────────────────────────
# OUTPUT FILTER
# ─────────────────────────────────────────────
# Patterns that should never appear in the final answer.
UNSAFE_PATTERNS = [
    # Exact dosage prescriptions
    (r"take\s+\d+\s*(mg|ml|tablets?|capsules?)", "[DOSAGE REDACTED — consult a pharmacist]"),
    # Self-diagnosis language
    (r"you (have|are suffering from|definitely have)", "[DIAGNOSIS REDACTED — see a doctor]"),
    # Confident medical advice
    (r"(you should|you must|you need to)\s+(stop|start|take|avoid)\s+\w+",
     "[ADVICE REDACTED — consult a healthcare professional]"),
]

def filter_output(text: str) -> str:
    """
    Scan the agent's final answer and redact any content that matches
    unsafe medical-advice patterns.

    Returns the sanitised text plus an appended disclaimer if any
    redactions were made.
    """
    redacted = False
    for pattern, replacement in UNSAFE_PATTERNS:
        new_text, n = re.subn(pattern, replacement, text, flags=re.IGNORECASE)
        if n:
            text = new_text
            redacted = True
    if redacted:
        text += (
            "\n\n⚠️  [GUARDRAIL] One or more sections were redacted because they "
            "resembled direct medical advice. Please consult a qualified healthcare "
            "professional for personalised guidance."
        )
    return text


# ─────────────────────────────────────────────
# RATE LIMITER
# ─────────────────────────────────────────────
class RateLimitedExecutor:
    """
    Wraps an AgentExecutor with:
      • max_tool_calls  – hard limit on LLM iterations per request.
      • requests_per_minute – sliding-window budget across all requests.

    Usage:
        safe_executor = RateLimitedExecutor(executor, max_tool_calls=10, requests_per_minute=5)
        response = safe_executor.invoke({"input": query})
    """

    def __init__(self, agent_executor, max_tool_calls: int = 10, requests_per_minute: int = 5):
        self.executor = agent_executor
        self.max_tool_calls = max_tool_calls
        self.rpm_limit = requests_per_minute
        self._window: deque = deque()          # timestamps of recent requests
        # Patch max_iterations on the underlying executor
        self.executor.max_iterations = max_tool_calls

    def _check_rate_limit(self):
        """Raise if the per-minute budget is exceeded."""
        now = time.time()
        # Drop timestamps older than 60 s
        while self._window and now - self._window[0] > 60:
            self._window.popleft()
        if len(self._window) >= self.rpm_limit:
            wait = 60 - (now - self._window[0])
            raise RuntimeError(
                f"[GUARDRAIL] Rate limit reached ({self.rpm_limit} req/min). "
                f"Please wait {wait:.0f} seconds before retrying."
            )
        self._window.append(now)

    def invoke(self, inputs: dict) -> dict:
        """Rate-check → run agent → filter output."""
        self._check_rate_limit()
        raw = self.executor.invoke(inputs)
        raw["output"] = filter_output(raw.get("output", ""))
        return raw


print("Guardrails loaded: output_filter + RateLimitedExecutor")


Guardrails loaded: output_filter + RateLimitedExecutor


In [22]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain import hub
from langsmith import Client

client = Client()

# Initialize the Language Model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


# Pull the ReAct prompt from LangChain Hub
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

# List of tools the agent can use
tools = [get_medical_info, get_drug_info, calculate_bmi, medical_disclaimer]

# Create the ReAct agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# Create the AgentExecutor
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=15,
    handle_parsing_errors=True
)

print("Medical Information Assistant Agent is ready")

# Wrap with guardrails
safe_executor = RateLimitedExecutor(executor, max_tool_calls=10, requests_per_minute=5)
print("Safe (guardrailed) executor is ready")


Medical Information Assistant Agent is ready
Safe (guardrailed) executor is ready


## 6. Full Execution Trace Example

Let's run a realistic medical query and observe the complete multi-step reasoning.

In [23]:
query = "I have been having frequent headaches and mild fever for the last 3 days. What could be the possible causes and what should I do?"

# Invoke through the guardrailed executor
try:
    response = safe_executor.invoke({"input": query})
    print("\n=== FINAL ANSWER FROM THE AGENT ===")
    print(response["output"])
except RuntimeError as e:
    print(e)




> Entering new AgentExecutor chain...
I should gather information about the possible causes of headaches and fever.
Action: get_medical_info
Action Input: "headache"Information about 'headache':

A headache, also known as cephalalgia, is the symptom of pain in the face, head, or neck. It can occur as a migraine, tension-type headache, or cluster headache.[1][2] There is an increased risk of depression in those with severe headaches.[3]

Headaches can occur as a result of many conditions. There are a number of different classification systems for headaches. The most well-recognized is that of the International Headache Society, which classifies it into more than 150 types of primary and secondary headaches. Causes of headaches may include dehydration; fatigue; sleep deprivation; stress;[4] the effects of medications (overuse) and recreational drugs, including withdrawal; viral infections; loud noises; head injury; rapid ingestion of a very cold food or beverage; and dental or sinus ..

## 7. Explanation of Tool Usage and Trace Analysis

In a typical execution, the agent follows this logical flow:

1. **Thought**: Understands the symptoms and decides to gather reliable medical information.
2. **Action**: Calls `get_medical_info` with the symptoms.
3. **Observation**: Receives information about possible causes.
4. **Thought**: May also call `medical_disclaimer` to ensure responsible response.
5. **Final Answer**: Provides helpful, balanced information with strong disclaimers.

This demonstrates true multi-step reasoning, responsible AI behavior, and safe tool usage.

**Key Observations:**
- The agent does not give direct medical advice
- It uses tools in a logical sequence
- It always includes disclaimers
- Real data is fetched from open sources (Wikipedia + Open-Meteo in previous tools)

## Conclusion

This Reference Agent System Walkthrough in the medical domain showed a complete, responsible, and production-like AI agent architecture.

This example demonstrates best practices for building meaningful and safe AI agents.

**Next Step:** Try modifying the query or adding new tools (e.g., vaccine information, allergy checker) to see how the system adapts.

## Resources

- [LangChain Documentation](https://python.langchain.com/docs/get_started/introduction)
- [ReAct Pattern Explained](https://www.promptingguide.ai/techniques/react)
- [BeautifulSoup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)